# Sputtering base pressure and Ba-target DC bias (Exp 7)

Read-only exploratory analysis of records visible to the authenticated NOMAD account. It explores which recorded run conditions vary with sputtering base pressure and whether the Ba target's recorded average voltage (RF self-bias where applicable) varies with cumulative target use.

The schema records `DTUSputtering.base_pressure` in Pa; each gun overview can record `target_material`, `target_usage` in kWh, `applied_power`, and `average_voltage` in V. The schema describes average voltage as DC bias for RF supplies. `DTUTarget.time_used` is a static lifetime value in seconds and is not the target-use state at a particular run. Neither run order nor static lifetime is treated as a direct erosion measurement.

Set `NOMAD_CLIENT_ACCESS_TOKEN` in the notebook runtime environment or, on Windows, run `setup_nomad_token.py` once to store it encrypted for your Windows account. Optionally override `NOMAD_BASE_URL`. Searches use `owner: visible`, which respects account permissions and does not grant access to other users' private entries. Metadata is queried before a bounded number of archives is fetched. Nothing is written back to NOMAD.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'notebooks' / 'oasis_trends'):
    if (candidate / 'oasis_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find notebooks/oasis_trends/oasis_helpers.py; start Jupyter in the repository or notebook folder.')

import pandas as pd
import plotly.express as px
from oasis_helpers import fetch_archives, reference_ids, search_entries

SPUTTER_SCHEMA = 'nomad_dtu_nanolab_plugin.schema_packages.sputtering.DTUSputtering'
TARGET_SCHEMA = 'nomad_dtu_nanolab_plugin.schema_packages.target.DTUTarget'
MAX_SEARCH_ENTRIES = 5000  # sufficient to cover all currently visible entries per schema
MAX_SPUTTER_ARCHIVES = 1000
MAX_TARGET_ARCHIVES = 1000
# Archive values are SI (Pa, K); pressures are reported in Torr and temperatures in degC.
PA_PER_TORR = 101325 / 760

## Search and fetch bounded archive sets

The limits below cap downloads, not the server's total number of matches. A capped set is an exploratory sample, so increase the limits only as needed and report the final counts.

In [ ]:
sputter_meta = search_entries(SPUTTER_SCHEMA, MAX_SEARCH_ENTRIES)
target_meta = search_entries(TARGET_SCHEMA, MAX_SEARCH_ENTRIES)
sputter_archives = fetch_archives(sputter_meta, MAX_SPUTTER_ARCHIVES)
target_ids = set()
for _, run_data in sputter_archives:
    for gun in ('magkeeper3', 'magkeeper4', 'taurus'):
        overview = (run_data.get('deposition_parameters') or {}).get(gun)
        if isinstance(overview, dict):
            target_ids.update(reference_ids(overview.get('target_id')))
target_meta_by_id = {meta.get('entry_id'): meta for meta in target_meta}
linked_target_meta = [target_meta_by_id[eid] for eid in target_ids if eid in target_meta_by_id]
target_archives = fetch_archives(linked_target_meta, MAX_TARGET_ARCHIVES)
print(f'Sputtering matches fetched: {len(sputter_archives)} / at most {MAX_SEARCH_ENTRIES}')
print(f'Referenced visible target archives fetched: {len(target_archives)} / {len(linked_target_meta)}')
print(f'Archive caps: sputtering={MAX_SPUTTER_ARCHIVES}, targets={MAX_TARGET_ARCHIVES}')

## Extract run and target diagnostics

One run can include multiple active guns. The rows below preserve one row per run and gun rather than collapsing away target-specific bias or usage. The target ID is used only for a strict reference match to a target archive.

In [ ]:
target_by_id = {meta.get('entry_id'): data for meta, data in target_archives}
run_rows = []
for meta, data in sputter_archives:
    dep = data.get('deposition_parameters') or {}
    for gun in ('magkeeper3', 'magkeeper4', 'taurus'):
        overview = dep.get(gun)
        if not isinstance(overview, dict):
            continue
        target_ids = reference_ids(overview.get('target_id'))
        target_data = next((target_by_id[t] for t in target_ids if t in target_by_id), {})
        material = overview.get('target_material') or target_data.get('main_material')
        run_rows.append({
            'entry_id': meta.get('entry_id'),
            'entry_name': meta.get('entry_name'),
            'entry_create_time': meta.get('entry_create_time'),
            'gun': gun,
            'base_pressure_Torr': (lambda v: None if v is None else v / PA_PER_TORR)(data.get('base_pressure')),
            'target_material': material,
            'target_number': target_data.get('target_number'),
            'target_usage_kWh': overview.get('target_usage'),
            'average_voltage_V': overview.get('average_voltage'),
            'power_type': overview.get('power_type'),
            'applied_power_W': overview.get('applied_power'),
            'target_time_used_s': target_data.get('time_used'),
                'deposition_temperature_degC': (lambda v: None if v is None else v - 273.15)(dep.get('deposition_temperature')),
            'deposition_pressure_Torr': (lambda v: None if v is None else v / PA_PER_TORR)(dep.get('deposition_pressure')),
            'uses_s_cracker': isinstance(dep.get('s_cracker'), dict),
            'target_reference_resolved': bool(target_ids & target_by_id.keys()),
        })
runs = pd.DataFrame(run_rows)
print(f'Run/gun rows: {len(runs)}')
if runs.empty:
    raise ValueError('No per-gun deposition overview rows found. Inspect schema versions and archive data before continuing.')
print('Missing-value fraction:')
display(runs.isna().mean().sort_values(ascending=False).rename('fraction_missing').to_frame())
print('Unresolved target references:', int((~runs['target_reference_resolved']).sum()))
display(runs.head())

## Base-pressure exploration

These plots describe recorded associations only. Pressure is converted from the archive's SI value (Pa) to Torr and temperature from K to degC. The factors available in this initial pass are target material, recorded deposition pressure/temperature, and whether an S cracker overview is present.

In [ ]:
pressure_data = runs.dropna(subset=['base_pressure_Torr']).copy()
print(f'Runs with base pressure: {len(pressure_data)} / {len(runs)}')
if pressure_data.empty:
    print('No base-pressure values were returned; check field coverage and archive permissions.')
else:
    display(px.box(pressure_data, x='target_material', y='base_pressure_Torr', color='uses_s_cracker',
                   points='all', hover_data=['entry_name', 'gun'],
                   title='Recorded base pressure by target material and S-cracker overview').show())
    temperature_data = pressure_data.dropna(subset=['deposition_temperature_degC'])
    if not temperature_data.empty:
        display(px.scatter(temperature_data, x='deposition_temperature_degC', y='base_pressure_Torr',
                           color='target_material', hover_data=['entry_name', 'gun'],
                           title='Base pressure versus recorded deposition temperature').show())
    display(pressure_data[['base_pressure_Torr', 'deposition_temperature_degC', 'deposition_pressure_Torr']].describe())

## Run timeline: base pressure and target changes

Each point is one sputtering entry, ordered by NOMAD entry creation time. The plotted target is the set of target materials recorded for that run. A change in the displayed target set marks a target change in the available record; this is not a verified physical replacement unless the target identity/number also changes.

In [ ]:
timeline_source = runs.dropna(subset=['entry_create_time']).copy()
if timeline_source.empty:
    print('No entry creation timestamps returned; cannot order runs reliably.')
else:
    timeline_source['run_time'] = pd.to_datetime(timeline_source['entry_create_time'], utc=True, errors='coerce')
    timeline_source = timeline_source.dropna(subset=['run_time'])
    timeline = timeline_source.groupby('entry_id', as_index=False).agg(
        run_time=('run_time', 'first'), entry_name=('entry_name', 'first'),
        base_pressure_Torr=('base_pressure_Torr', 'first'),
        target_set=('target_material', lambda values: ' + '.join(sorted({str(v) for v in values.dropna()})) or 'unknown'),
        target_numbers=('target_number', lambda values: ', '.join(sorted({str(x) for x in values.dropna()})) or 'unknown'))
    timeline = timeline.sort_values(['run_time', 'entry_id']).reset_index(drop=True)
    timeline['target_changed'] = timeline['target_set'].ne(timeline['target_set'].shift())
    timeline.loc[timeline.index[0], 'target_changed'] = False
    timeline['is_zr_run'] = timeline['target_set'].str.contains(r'(?<![A-Za-z])Zr(?![a-z])', case=False, regex=True)
    timeline['zr_sputtered_before'] = timeline['is_zr_run'].shift(fill_value=False).cummax()
    display(timeline[['run_time',     'entry_name', 'base_pressure_Torr', 'target_set', 'target_numbers', 'target_changed', 'is_zr_run']])
    display(px.scatter(timeline.dropna(subset=['base_pressure_Torr']), x='run_time', y='base_pressure_Torr',
                        color='target_set', symbol='target_changed', hover_data=['entry_name', 'target_numbers', 'is_zr_run'],
                        title='Sputtering run timeline: base pressure and recorded target changes').show())
    display(px.scatter(timeline, x='run_time', y='target_set', color='target_set', symbol='target_changed',
                        hover_data=['entry_name', 'target_numbers'],
                        title='Target material by sputtering run chronology').show())
    # A run's base pressure is measured before that deposition. Exclude Zr runs
    # themselves when comparing subsequent non-Zr runs to earlier non-Zr runs.
    context = timeline[~timeline['is_zr_run'] & timeline['base_pressure_Torr'].notna()].copy()
    context['period'] = context['zr_sputtered_before'].map({False: 'Before first Zr run', True: 'After a prior Zr run'})
    display(context.groupby('period')['base_pressure_Torr'].agg(['count', 'median', 'mean', 'std']))
    print('This before/after summary is chronological and exploratory; recipe, target, maintenance and time trends can confound it.')

## Exp 7: is Ba-target average voltage associated with target usage?

The schema's `target_usage` is run-specific accumulated energy at the end of a deposition (kWh), so it is a potential within-target usage proxy. The comparison is restricted to records identified as Ba by the target material string; inspect the table before interpreting inclusion. Average voltage is not necessarily an RF bias for every power type. Signed voltage and absolute magnitude answer different meanings of “goes down.”

In [ ]:
ba_mask = runs['target_material'].fillna('').astype(str).str.contains(r'(?<![A-Za-z])Ba(?![a-z])', case=False, regex=True)
ba = runs.loc[ba_mask].copy()
print(f'Rows identified as Ba target: {len(ba)}')
if ba.empty:
    print('No target material strings matched Ba. Review target reference coverage and naming before concluding there are no Ba runs.')
else:
    display(ba[['entry_name', 'gun', 'target_material', 'target_number', 'target_usage_kWh',
                'average_voltage_V', 'power_type', 'applied_power_W', 'target_time_used_s']])
    usage_bias = ba.dropna(subset=['target_usage_kWh', 'average_voltage_V']).copy()
    if usage_bias.empty:
        print('No Ba rows contain both run-specific target_usage and average_voltage.')
    else:
        usage_bias['absolute_voltage_V'] = usage_bias['average_voltage_V'].abs()
        display(px.scatter(usage_bias, x='target_usage_kWh', y='average_voltage_V', color='gun',
                           symbol='power_type', size='applied_power_W', hover_data=['entry_name'],
                           title='Ba target: recorded average voltage versus run-specific accumulated use').show())
        display(px.scatter(usage_bias, x='target_usage_kWh', y='absolute_voltage_V', color='gun',
                           hover_data=['entry_name', 'applied_power_W'],
                           title='Ba target: |average voltage| versus run-specific accumulated use').show())
    static = ba.dropna(subset=['target_time_used_s', 'average_voltage_V'])
    if not static.empty:
        display(px.scatter(static, x='target_time_used_s', y='average_voltage_V', color='target_number',
                           hover_data=['entry_name', 'target_usage_kWh'],
                           title='Ba target: average voltage versus static target lifetime (between-target comparison)').show())
        print('Static time_used is not a per-run erosion trajectory; interpret only as a between-target comparison.')

## Interpretation checks

- Compare voltage only within the same target/gun and similar applied-power conditions where possible. Power, gas composition, pressure, target conditioning, and process recipe can confound the trend.
- `target_usage` is an accumulated-energy proxy, not a direct erosion-depth measurement. Verify it is monotonic and measured consistently for the Ba target across runs.
- `time_used` is a static target-entry lifetime value; it does not state how eroded the target was at the time of each run.
- A single replaced Ba target may give only one target trajectory. Report the number of distinct target IDs, matched runs, and missing values before any trend estimate.
- The base-pressure plots do not establish causes. Cleaning, venting, downtime, pump state, and chamber history may be missing from these records.
- All conclusions are limited to records visible to this token and the selected archive caps. No data have been written to NOMAD.